In [1]:
import pandas as pd
import os

In [2]:
processed_path = os.path.join('..', 'data', 'processed')

## Idealista Price m2: 08-2016 até 11-2025 (1904 observações)

In [3]:
path_idealista = os.path.join(processed_path, 'Lisboa_freguesias_preco_m2_long_imputed.csv')

target = pd.read_csv(path_idealista)

In [4]:
target

,date,Year,Month,parish,price_m2,imputed_flag
0,2016-08-01,2016,8,Alvalade,11.100000,0
1,2016-08-01,2016,8,Areeiro,11.100000,0
2,2016-08-01,2016,8,Arroios,11.800000,0
3,2016-08-01,2016,8,Avenidas Novas,12.100000,0
4,2016-08-01,2016,8,Belém,10.433333,1
...,...,...,...,...,...,...
1899,2025-11-01,2025,11,Penha de França,20.000000,0
1900,2025-11-01,2025,11,Santa Maria Maior,27.900000,0
1901,2025-11-01,2025,11,Santo António,28.300000,0
1902,2025-11-01,2025,11,São Domingos de Benfica,17.800000,0


## POIS: Static variables

In [5]:
pois = pd.read_csv(os.path.join(processed_path, 'POIS_freguesias_lisboa.csv'))
pois = pois.rename(columns={'freguesia': 'parish'})

# merge only by Freguesia because POIS dataset dows not have month or year, is static for the parishes
merged_data = pd.merge(target, pois, on='parish', how='left')
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Data columns (total 27 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   date                                            1904 non-null   object 
 1   Year                                            1904 non-null   int64  
 2   Month                                           1904 non-null   int64  
 3   parish                                          1904 non-null   object 
 4   price_m2                                        1904 non-null   float64
 5   imputed_flag                                    1904 non-null   int64  
 6   Num_Estacoes_Metro                              1904 non-null   int64  
 7   Num_Estacoes_Comboio                            1904 non-null   int64  
 8   Num_Prestadores_Saude                           1904 non-null   int64  
 9   Area_km2                                 

## VIIRS - Monthly Mean Nighttime Radiance
Proxy for nocturnal economic and tourist activity intensity.

In [6]:
viirs = pd.read_csv(os.path.join(processed_path, 'VIIRS_Lisboa_Freguesias_2015_2025.csv'))
viirs = viirs.rename(columns={'Freguesia': 'parish', "mean": "MeanNightimeRadianceVIIRS"})
viirs['Year'] = viirs['Year'].astype(int)
viirs['Month'] = viirs['Month'].astype(int)

In [7]:
merged_data = pd.merge(merged_data, viirs[['parish', 'Year', 'Month', 'MeanNightimeRadianceVIIRS']], on=['parish', 'Year', 'Month'], how='left')
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Data columns (total 28 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   date                                            1904 non-null   object 
 1   Year                                            1904 non-null   int64  
 2   Month                                           1904 non-null   int64  
 3   parish                                          1904 non-null   object 
 4   price_m2                                        1904 non-null   float64
 5   imputed_flag                                    1904 non-null   int64  
 6   Num_Estacoes_Metro                              1904 non-null   int64  
 7   Num_Estacoes_Comboio                            1904 non-null   int64  
 8   Num_Prestadores_Saude                           1904 non-null   int64  
 9   Area_km2                                 

## RJUE - Alvaras e Pedidos

In [8]:
rjue_alvaras = pd.read_csv(os.path.join(processed_path, 'RJUE_dados_urbanismo_alvaras.csv'))
rjue_pedidos = pd.read_csv(os.path.join(processed_path, 'RJUE_dados_urbanismo_pedidos.csv'))

In [9]:
freguesias_precos = set(merged_data['parish'].unique())
freguesias_rjue = set(rjue_pedidos['Freguesia'].unique())

# 2. Encontrar quem não tem par
apenas_no_preço = freguesias_precos - freguesias_rjue
apenas_no_rjue = freguesias_rjue - freguesias_precos

print(f"🚩 No Idealista mas não no RJUE: {apenas_no_preço}")
print(f"🚩 No RJUE mas não no Idealista: {apenas_no_rjue}")

🚩 No Idealista mas não no RJUE: {'São Domingos de Benfica', 'Parque das Nações', 'Penha de França', 'Campo de Ourique'}
🚩 No RJUE mas não no Idealista: {'São Domingos De Benfica', 'Campo De Ourique', 'Penha De França', 'Parque Das Nações'}


In [ ]:
mapa_freguesias = {
    'São Domingos De Benfica': 'São Domingos de Benfica',
    'Campo De Ourique': 'Campo de Ourique',
    'Parque Das Nações': 'Parque das Nações',
    'Penha De França': 'Penha de França'
}

# Aplicar a correção nos datasets RJUE
rjue_pedidos['Freguesia'] = rjue_pedidos['Freguesia'].replace(mapa_freguesias)
rjue_alvaras['Freguesia'] = rjue_alvaras['Freguesia'].replace(mapa_freguesias)

# Verificar se agora os conjuntos são idênticos
diff = set(merged_data['parish'].unique()) ^ set(rjue_pedidos['Freguesia'].unique())
if not diff:
    print("✅ Sucesso! As freguesias estão 100% sincronizadas.")
else:
    print(f"⚠️ Ainda existem diferenças: {diff}")

✅ Sucesso! As freguesias estão 100% sincronizadas.


In [11]:
df_pedidos_join = rjue_pedidos.rename(columns={'Freguesia': 'parish', 'AnoMes': 'date_key'})
df_alvaras_join = rjue_alvaras.rename(columns={'Freguesia': 'parish', 'AnoMes': 'date_key'})

# 2. PREPARAÇÃO DO DATASET PRINCIPAL (image_7ce641)
# Criamos uma 'date_key' (YYYY-MM) a partir de Year e Month para o merge ser exato
merged_data['date_key'] = (
    merged_data['Year'].astype(str) + '-' + 
    merged_data['Month'].astype(str).str.zfill(2)
)

In [12]:
# 3. O MEGA-MERGE (Left Join)
# Começamos pelo dataset de preços para não perder nenhuma linha do Idealista
final_df = pd.merge(merged_data, df_pedidos_join, on=['parish', 'date_key'], how='left')
final_df = pd.merge(final_df, df_alvaras_join, on=['parish', 'date_key'], how='left')

In [13]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Data columns (total 45 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   date                                            1904 non-null   object 
 1   Year                                            1904 non-null   int64  
 2   Month                                           1904 non-null   int64  
 3   parish                                          1904 non-null   object 
 4   price_m2                                        1904 non-null   float64
 5   imputed_flag                                    1904 non-null   int64  
 6   Num_Estacoes_Metro                              1904 non-null   int64  
 7   Num_Estacoes_Comboio                            1904 non-null   int64  
 8   Num_Prestadores_Saude                           1904 non-null   int64  
 9   Area_km2                                 

## RNAL

In [14]:
# Merge with RNAL_variaveis_mensaias_freguesias_2008_2025.csv
rnals = pd.read_csv(os.path.join(processed_path, 'RNAL_dataset_final.csv'))
rnals = rnals.rename(columns={'Freguesia': 'parish', 'Ano': 'Year', 'Mes': 'Month'})
# Levas apenas o stock histórico e as métricas de pressão.
colunas_al = [
    'parish', 'Year', 'Month', 
    'Densidade_AL_km2', 'Densidade_Capacidade_km2',
    'capacidade_media_por_al', 'al_growth_rate', 'racio_mercado_informal'
]
rnals = rnals[colunas_al]
rnals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5112 entries, 0 to 5111
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   parish                    5112 non-null   object 
 1   Year                      5112 non-null   int64  
 2   Month                     5112 non-null   int64  
 3   Densidade_AL_km2          5112 non-null   float64
 4   Densidade_Capacidade_km2  5112 non-null   float64
 5   capacidade_media_por_al   5112 non-null   float64
 6   al_growth_rate            5112 non-null   float64
 7   racio_mercado_informal    5112 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 319.6+ KB


In [15]:
# Merge the datasets on 'parish', 'Year', and 'Month'
merged_data = pd.merge(final_df, rnals, on=['parish', 'Year', 'Month'], how='left')
merged_data

,date,Year,Month,parish,price_m2,imputed_flag,Num_Estacoes_Metro,Num_Estacoes_Comboio,Num_Prestadores_Saude,Area_km2,...,Stock_12M_ALV_Reabilitacao_Count,Stock_12M_ALV_Demolicao_Area,Stock_12M_ALV_Demolicao_Count,Racio_Gentrificacao_ALV_Area,Racio_Gentrificacao_ALV_Count,Densidade_AL_km2,Densidade_Capacidade_km2,capacidade_media_por_al,al_growth_rate,racio_mercado_informal
0,2016-08-01,2016,8,Alvalade,11.100000,0,3,1,4,5.34,...,25.0,0.000000,0.0,1.079371e+01,24.997500,5.430712,43.632959,8.034483,0.260870,0.000000
1,2016-08-01,2016,8,Areeiro,11.100000,0,1,0,0,1.74,...,12.0,0.000000,0.0,4.415204e+07,120000.000000,27.011494,282.183908,10.446809,0.000000,0.148936
2,2016-08-01,2016,8,Arroios,11.800000,0,4,0,6,2.13,...,44.0,561.459961,1.0,2.534457e+01,21.998900,205.633803,1816.901408,8.835616,0.050360,0.013699
3,2016-08-01,2016,8,Avenidas Novas,12.100000,0,6,1,5,2.99,...,47.0,0.000000,0.0,3.175345e+01,23.498825,38.795987,494.648829,12.750000,0.195876,0.008621
4,2016-08-01,2016,8,Belém,10.433333,1,0,1,1,10.43,...,33.0,885.103027,11.0,5.005450e+00,4.124948,5.656759,30.009588,5.305085,0.053571,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,2025-11-01,2025,11,Penha de França,20.000000,0,0,0,2,2.71,...,1.0,0.000000,0.0,1.253799e+06,10000.000000,213.653137,1278.966790,5.986183,0.000000,0.006908
1900,2025-11-01,2025,11,Santa Maria Maior,27.900000,0,5,1,2,3.01,...,11.0,0.000000,0.0,4.400366e+07,110000.000000,1438.205980,7757.475083,5.393855,0.000000,0.011550
1901,2025-11-01,2025,11,Santo António,28.300000,0,3,0,6,1.49,...,11.0,0.000000,0.0,1.100448e+08,110000.000000,1024.832215,6693.959732,6.531762,0.000000,0.035363
1902,2025-11-01,2025,11,São Domingos de Benfica,17.800000,0,3,1,5,4.29,...,0.0,0.000000,0.0,0.000000e+00,0.000000,38.461538,245.687646,6.387879,0.000000,0.000000


In [16]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Data columns (total 50 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   date                                            1904 non-null   object 
 1   Year                                            1904 non-null   int64  
 2   Month                                           1904 non-null   int64  
 3   parish                                          1904 non-null   object 
 4   price_m2                                        1904 non-null   float64
 5   imputed_flag                                    1904 non-null   int64  
 6   Num_Estacoes_Metro                              1904 non-null   int64  
 7   Num_Estacoes_Comboio                            1904 non-null   int64  
 8   Num_Prestadores_Saude                           1904 non-null   int64  
 9   Area_km2                                 

## CENSOS - Static Variable

In [17]:
censos = pd.read_csv(os.path.join(processed_path, 'CENSOS_dados.csv'))
censos = censos.rename(columns={'Freguesia': 'parish'})
merged_data = pd.merge(merged_data, censos, on='parish', how='left')
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Data columns (total 85 columns):
 #   Column                                                                              Non-Null Count  Dtype  
---  ------                                                                              --------------  -----  
 0   date                                                                                1904 non-null   object 
 1   Year                                                                                1904 non-null   int64  
 2   Month                                                                               1904 non-null   int64  
 3   parish                                                                              1904 non-null   object 
 4   price_m2                                                                            1904 non-null   float64
 5   imputed_flag                                                                        1904 non-null

In [18]:
merged_data.to_csv(os.path.join(processed_path, 'final_dataset.csv'), index=False)